# 4.2 🏗️ Arquitectura de Datos: El Patrón Repositorio

En proyectos reales de Inteligencia de Negocios y Analítica, **la obtención de datos no es trivial**.

Consumimos información desde:

- APIs financieras
- Data Warehouses
- Data Lakes
- Archivos planos (CSV / Parquet)
- Servicios pagos (Bloomberg, Refinitiv)
- Bases transaccionales

Si mezclamos esta lógica dentro de nuestros modelos de negocio, el sistema se vuelve:

- Difícil de mantener
- Costoso de migrar
- Acoplado a proveedores
- Difícil de testear

Para resolver esto usamos un patrón de arquitectura clásico:

> **El Patrón Repositorio (Repository / DataSource Pattern)**  


## 🧠 Problema que resuelve

Imagina que hoy usas **Yahoo Finance** y mañana:

- Migras a Bloomberg
- Compras datos históricos
- Pasas a BigQuery
- Integras un Feature Store

Si tu código está lleno de `yfinance`, tendrías que reescribir:

- Modelos
- Servicios
- Cálculos
- Tests

El patrón repositorio desacopla la fuente de datos de la lógica de negocio.


## 🧭 Arquitectura conceptual

```{mermaid}
flowchart LR
    A[Modelos de Negocio] --> B[MarketDataProvider Interface]
    B --> C[Yahoo Finance Client]
    B --> D[Bloomberg Client]
    B --> E[BigQuery Client]
    B --> F[CSV Client]
```


## 🧩 La Interfaz Abstracta

Definimos un contrato común para cualquier proveedor de datos.

Todos los clientes deben implementar los mismos métodos.

```python
# src/smart_portfolio/interfaces.py
from abc import ABC, abstractmethod
import pandas as pd

class MarketDataProvider(ABC):

    @abstractmethod
    def obtener_precio_actual(self, ticker: str) -> float:
        pass

    @abstractmethod
    def obtener_historia(
        self,
        ticker: str,
        dias: int
    ) -> pd.DataFrame:
        pass
```


### 🎯 Beneficios de la interfaz

- Estandariza consumo de datos
- Permite cambiar proveedor sin romper modelos
- Facilita testing (mocks / stubs)
- Habilita inyección de dependencias


## 📡 Implementación 1: Yahoo Finance

Primero instalamos la librería:

```bash
poetry add yfinance
```

Luego implementamos el cliente:


In [ ]:
import pandas as pd
import yfinance as yf

class YahooFinanceClient:

    def obtener_precio_actual(self, ticker: str) -> float:
        try:
            t = yf.Ticker(ticker)
            return float(t.fast_info["last_price"])
        except Exception as e:
            raise ConnectionError(
                f"Error conectando a Yahoo Finance: {e}"
            )

    def obtener_historia(
        self,
        ticker: str,
        dias: int = 365
    ) -> pd.DataFrame:

        t = yf.Ticker(ticker)

        # Simplificación: siempre 1 año
        df = t.history(period="1y")

        return df


## 🧪 Ejemplo de uso

Nuestros modelos no saben de Yahoo.  
Solo conocen la interfaz conceptual.


In [ ]:
provider = YahooFinanceClient()

precio = provider.obtener_precio_actual("AAPL")
print("Precio actual:", precio)

hist = provider.obtener_historia("AAPL")
print(hist.head())


## 🔄 Flujo completo de consumo

```{mermaid}
sequenceDiagram
    participant Modelo
    participant Interface
    participant YahooClient
    participant API

    Modelo->>Interface: obtener_precio("AAPL")
    Interface->>YahooClient: delega llamada
    YahooClient->>API: request datos
    API-->>YahooClient: JSON precios
    YahooClient-->>Modelo: float precio
```


## 🧪 Testing: Mock Provider

Una gran ventaja es que podemos testear sin internet.


In [ ]:
class MockMarketDataProvider:

    def obtener_precio_actual(self, ticker: str) -> float:
        return 100.0

    def obtener_historia(self, ticker: str, dias: int):
        import pandas as pd

        return pd.DataFrame({
            "Close": [100, 101, 102]
        })


mock = MockMarketDataProvider()

print(mock.obtener_precio_actual("AAPL"))
print(mock.obtener_historia("AAPL", 3))


## 🚀 Ventajas arquitectónicas

- Desacoplamiento proveedor ↔ negocio
- Migración tecnológica barata
- Testeo sin dependencias externas
- Reutilización de clientes
- Escalabilidad multi‑fuente

---

## 🎯 Cierre conceptual

El Patrón Repositorio convierte las fuentes de datos en **componentes intercambiables**.

Tus modelos dejan de depender de:

- APIs específicas
- Librerías externas
- Formatos particulares

Y pasan a depender de un contrato estable.

Eso es arquitectura profesional de datos 🏗️
